In [ ]:
# Conectar a la base de datos
conn = sqlite3.connect("proyecto_bic.db")

# Cargar los dos archivos de imagen
path_mamo = r"C:\Users\yerodri\Downloads\Ejercicios BIC\PFC\mamografias_ML.xlsx"
path_birads = r"C:\Users\yerodri\Downloads\Ejercicios BIC\PFC\imagenes_informes_BI-RADS.xlsx"

df_mamo = pd.read_excel(path_mamo)
df_birads = pd.read_excel(path_birads)

# Limpiar mamografias (igual que en Notebook 1)
df_mamo = df_mamo.dropna(how='all')
df_mamo = df_mamo.dropna(subset=['patient'])
df_mamo = df_mamo.drop(columns=['Fecha'])

# Corregir errores tipográficos en diagnóstico
df_mamo['diagnostic'] = df_mamo['diagnostic'].replace({
    'Banign': 'Benign',
    'Molignant': 'Malignant'
})

# Limpiar BI-RADS (quedarnos solo con filas que tienen patient)
df_birads = df_birads.dropna(subset=['patient'])

# Fusionar (left join para conservar todas las mamografías)
df_radiologia = df_mamo.merge(df_birads, on='patient', how='left')

# Renombrar columnas para SQL
df_radiologia = df_radiologia.rename(columns={
    'patient': 'patient_id'
})

# Crear tabla en SQL
df_radiologia.to_sql('RADIOLOGIA', conn, if_exists='replace', index=False)

print(f"✅ Tabla RADIOLOGIA creada con {df_radiologia.shape[0]} filas y {df_radiologia.shape[1]} columnas")
print("\n--- Muestra de RADIOLOGIA ---")
print(df_radiologia.head())

# Cerrar conexión (la reabriremos después)
conn.close()

consultas SQL: hacemos 5 consultas que integran las 3 tablas, como pide el trabajo.

In [2]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("proyecto_bic.db")

Consulta 1: Pacientes con BI-RADS 5 y mutación BRCA1
*"Dime las pacientes con BI-RADS 5 que además tengan la mutación BRCA1"*

In [3]:
query1 = '''
SELECT DISTINCT p.patient_id, p.sex, p.hospital, r."Categoría BI-RADS", g.variant_id
FROM PACIENTES p
JOIN RADIOLOGIA r ON p.patient_id = r.patient_id
JOIN GENOMICA g ON p.patient_id = g.patient_id
WHERE r."Categoría BI-RADS" = 'BI-RADS 5'
  AND g.variant_id = 'rs1000'
  AND g.genotype IN ('0/1', '1/0', '1/1')
LIMIT 20
'''

df1 = pd.read_sql_query(query1, conn)
print(f"✅ Pacientes con BI-RADS 5 y mutación: {len(df1)}")
display(df1)

✅ Pacientes con BI-RADS 5 y mutación: 11


,patient_id,sex,hospital,Categoría BI-RADS,variant_id
0,p22,Female,Hospital del Mar,BI-RADS 5,rs1000
1,p136,Female,Hospital del Mar,BI-RADS 5,rs1000
2,p147,Female,Hospital Clínic de Barcelona,BI-RADS 5,rs1000
3,p182,Female,Hospital Clínic de Barcelona,BI-RADS 5,rs1000
4,p269,Female,Hospital Clínic de Barcelona,BI-RADS 5,rs1000
5,p305,Female,Hospital Clínic de Barcelona,BI-RADS 5,rs1000
6,p374,Female,Hospital Clínic de Barcelona,BI-RADS 5,rs1000
7,p447,Female,Hospital Clínic de Barcelona,BI-RADS 5,rs1000
8,p501,Female,Hospital del Mar,BI-RADS 5,rs1000
9,p502,Female,Hospital Universitari Dr. Josep Trueta,BI-RADS 5,rs1000


Consulta 2: Relación entre diagnóstico y tamaño del tumor

In [4]:
query2 = '''
SELECT 
    r.diagnostic,
    AVG(r.radius) as radio_promedio,
    AVG(r.area) as area_promedio,
    COUNT(*) as total_pacientes
FROM RADIOLOGIA r
WHERE r.diagnostic IS NOT NULL
GROUP BY r.diagnostic
'''

df2 = pd.read_sql_query(query2, conn)
print("✅ Comparación diagnóstico vs tamaño")
display(df2)

✅ Comparación diagnóstico vs tamaño


,diagnostic,radio_promedio,area_promedio,total_pacientes
0,Benign,12.221290,477.220613,359
1,Malignant,18.096542,973.738318,214


Consulta 3: Pacientes con alta frecuencia de variantes genéticas

In [5]:
query3 = '''
SELECT 
    p.patient_id,
    COUNT(g.variant_id) as num_variantes,
    SUM(CASE WHEN g.genotype IN ('0/1', '1/0', '1/1') THEN 1 ELSE 0 END) as mutaciones_detectadas
FROM PACIENTES p
JOIN GENOMICA g ON p.patient_id = g.patient_id
GROUP BY p.patient_id
ORDER BY mutaciones_detectadas DESC
LIMIT 10
'''

df3 = pd.read_sql_query(query3, conn)
print("✅ Top 10 pacientes con más mutaciones")
display(df3)

✅ Top 10 pacientes con más mutaciones


,patient_id,num_variantes,mutaciones_detectadas
0,p389,2500,1317
1,p132,2500,1309
2,p373,2500,1308
3,p490,2500,1307
4,p290,2500,1306
5,p108,2500,1306
6,p533,2500,1305
7,p381,2500,1304
8,p327,2500,1304
9,p191,2500,1303


Consulta 4: Distribución de BI-RADS por hospital

In [6]:
query4 = '''
SELECT 
    p.hospital,
    r."Categoría BI-RADS",
    COUNT(*) as total
FROM PACIENTES p
JOIN RADIOLOGIA r ON p.patient_id = r.patient_id
WHERE r."Categoría BI-RADS" IS NOT NULL
GROUP BY p.hospital, r."Categoría BI-RADS"
ORDER BY p.hospital, r."Categoría BI-RADS"
'''

df4 = pd.read_sql_query(query4, conn)
print("✅ Distribución BI-RADS por hospital")
display(df4)

✅ Distribución BI-RADS por hospital


,hospital,Categoría BI-RADS,total
0,Hospital Clínic de Barcelona,BI-RADS 3,3
1,Hospital Clínic de Barcelona,BI-RADS 4A,13
2,Hospital Clínic de Barcelona,BI-RADS 4B,6
3,Hospital Clínic de Barcelona,BI-RADS 4C,5
4,Hospital Clínic de Barcelona,BI-RADS 5,10
5,Hospital Universitari Arnau de Vilanova,BI-RADS 3,1
6,Hospital Universitari Dr. Josep Trueta,BI-RADS 3,6
7,Hospital Universitari Dr. Josep Trueta,BI-RADS 4A,2
8,Hospital Universitari Dr. Josep Trueta,BI-RADS 4B,2
9,Hospital Universitari Dr. Josep Trueta,BI-RADS 4C,1


Consulta 5: Perfil genético de pacientes con diagnóstico Malignant

In [7]:
query5 = '''
SELECT 
    r.diagnostic,
    g.variant_id,
    COUNT(*) as frecuencia,
    SUM(CASE WHEN g.genotype IN ('0/1', '1/0', '1/1') THEN 1 ELSE 0 END) as mutaciones
FROM RADIOLOGIA r
JOIN GENOMICA g ON r.patient_id = g.patient_id
WHERE r.diagnostic = 'Malignant'
  AND g.genotype != './.'
GROUP BY g.variant_id
ORDER BY mutaciones DESC
LIMIT 10
'''

df5 = pd.read_sql_query(query5, conn)
print("✅ Variantes más frecuentes en pacientes Malignant")
display(df5)

✅ Variantes más frecuentes en pacientes Malignant


,diagnostic,variant_id,frecuencia,mutaciones
0,Malignant,rs2172,171,133
1,Malignant,rs2917,177,129
2,Malignant,rs3342,167,128
3,Malignant,rs3415,167,127
4,Malignant,rs3124,167,127
5,Malignant,rs2964,170,127
6,Malignant,rs2877,176,127
7,Malignant,rs2404,178,127
8,Malignant,rs1767,169,127
9,Malignant,rs1576,167,127


In [8]:
conn.close()
print("🏁 Consultas finalizadas")

🏁 Consultas finalizadas
